# Notebook 06: Cryptocurrency Market Analysis

## Overview

This notebook provides a comprehensive introduction to cryptocurrency market analysis, covering technical indicators, on-chain metrics, correlation analysis, and portfolio optimization. All analysis uses **synthetic data** generated with realistic statistical properties, making this notebook fully reproducible and executable offline.

By the end of this notebook, you will understand the quantitative tools used by institutional analysts and portfolio managers to evaluate cryptocurrency markets.

## Prerequisites
- Basic Python programming
- Familiarity with pandas DataFrames and Series
- Elementary statistics (mean, standard deviation, correlation)

## Learning Objectives
1. Generate realistic synthetic cryptocurrency price data using geometric Brownian motion
2. Implement core technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands) from scratch
3. Calculate and interpret on-chain metrics such as NVT Ratio
4. Perform correlation analysis across multiple crypto assets
5. Construct efficient frontiers and identify optimal portfolios using Modern Portfolio Theory

## Estimated Time: 4--6 hours

## Related Materials
- See `sections/04-blockchain-economics.md` for theoretical foundations on crypto-asset valuation

---
## Setup

Import all required libraries. This notebook uses only standard scientific Python packages and requires **no API keys or internet access**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from tqdm.auto import tqdm

# Reproducibility
np.random.seed(42)

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print('All libraries loaded successfully.')

---
# Part 1: Generating Realistic Market Data

Real cryptocurrency price data exhibits several stylized facts: heavy tails, volatility clustering, and cross-asset correlation. We use **Geometric Brownian Motion (GBM)** with correlated Wiener processes to generate synthetic data that captures these properties.

The GBM model for a price $S_t$ is:

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

In discrete time this becomes:

$$S_{t+1} = S_t \exp\left[(\mu - \tfrac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} \, Z_t\right]$$

where $Z_t \sim N(0,1)$.

### 1.1 Generating Correlated BTC and ETH Close Prices

In [ ]:
def generate_gbm_prices(S0, mu, sigma, T, dt, n_paths=1, random_state=None):
    """
    Generate price paths using Geometric Brownian Motion.
    
    Parameters
    ----------
    S0 : float        - Initial price
    mu : float        - Annual drift (expected return)
    sigma : float     - Annual volatility
    T : float         - Time horizon in years
    dt : float        - Time step in years (1/365 for daily)
    n_paths : int     - Number of independent paths
    random_state : np.random.Generator or None
    
    Returns
    -------
    np.ndarray of shape (n_steps+1, n_paths)
    """
    rng = random_state or np.random.default_rng(42)
    n_steps = int(T / dt)
    Z = rng.standard_normal((n_steps, n_paths))
    drift = (mu - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * Z
    log_returns = drift + diffusion
    log_prices = np.vstack([np.zeros((1, n_paths)), np.cumsum(log_returns, axis=0)])
    prices = S0 * np.exp(log_prices)
    return prices

print('GBM generator defined.')

In [ ]:
# Parameters
T = 2.0          # 2 years
dt = 1 / 365     # daily
n_days = int(T / dt)

# BTC parameters
S0_btc = 30_000
mu_btc = 0.15     # 15% annual drift
sigma_btc = 0.60  # 60% annual volatility

# ETH parameters
S0_eth = 2_000
mu_eth = 0.20     # 20% annual drift
sigma_eth = 0.80  # 80% annual volatility

# Correlation between BTC and ETH
rho = 0.70

# Generate correlated normal random variables via Cholesky decomposition
rng = np.random.default_rng(42)
Z_independent = rng.standard_normal((n_days, 2))

# Cholesky factor for [[1, rho], [rho, 1]]
L = np.array([[1.0, 0.0],
              [rho, np.sqrt(1 - rho**2)]])

Z_correlated = Z_independent @ L.T  # shape (n_days, 2)

# BTC prices
drift_btc = (mu_btc - 0.5 * sigma_btc**2) * dt
log_ret_btc = drift_btc + sigma_btc * np.sqrt(dt) * Z_correlated[:, 0]
btc_close = S0_btc * np.exp(np.concatenate([[0], np.cumsum(log_ret_btc)]))

# ETH prices
drift_eth = (mu_eth - 0.5 * sigma_eth**2) * dt
log_ret_eth = drift_eth + sigma_eth * np.sqrt(dt) * Z_correlated[:, 1]
eth_close = S0_eth * np.exp(np.concatenate([[0], np.cumsum(log_ret_eth)]))

print(f'Generated {len(btc_close)} daily prices for BTC and ETH.')
print(f'BTC range: ${btc_close.min():,.0f} -- ${btc_close.max():,.0f}')
print(f'ETH range: ${eth_close.min():,.0f} -- ${eth_close.max():,.0f}')

### 1.2 Building OHLCV DataFrames

We construct realistic Open, High, Low, Close, and Volume data from the close prices. Intraday variation is simulated with small random offsets.

In [ ]:
def build_ohlcv(close_prices, base_volume, vol_noise_pct=0.3, seed=42):
    """
    Build a realistic OHLCV DataFrame from close prices.
    
    Open ~ previous close with small noise.
    High = max(open, close) * (1 + small positive noise).
    Low  = min(open, close) * (1 - small positive noise).
    Volume is log-normally distributed and slightly correlated with |return|.
    """
    rng = np.random.default_rng(seed)
    n = len(close_prices)
    dates = pd.date_range(start='2024-01-01', periods=n, freq='D')
    
    close = close_prices.copy()
    
    # Open: previous close with small noise
    open_prices = np.empty(n)
    open_prices[0] = close[0] * (1 + rng.normal(0, 0.002))
    open_prices[1:] = close[:-1] * (1 + rng.normal(0, 0.002, size=n-1))
    
    # High and Low
    intraday_range = np.abs(rng.normal(0, 0.015, size=n))
    high = np.maximum(open_prices, close) * (1 + intraday_range)
    low = np.minimum(open_prices, close) * (1 - intraday_range)
    
    # Volume: log-normal, correlated with absolute return
    returns = np.zeros(n)
    returns[1:] = np.abs(np.diff(np.log(close)))
    vol_signal = returns / (returns.mean() + 1e-10)  # normalized
    log_vol = np.log(base_volume) + 0.3 * vol_signal + rng.normal(0, vol_noise_pct, size=n)
    volume = np.exp(log_vol).astype(int)
    
    df = pd.DataFrame({
        'Open': open_prices,
        'High': high,
        'Low': low,
        'Close': close,
        'Volume': volume
    }, index=dates)
    df.index.name = 'Date'
    return df

btc_df = build_ohlcv(btc_close, base_volume=25_000, seed=42)
eth_df = build_ohlcv(eth_close, base_volume=500_000, seed=43)

print('BTC OHLCV DataFrame:')
btc_df.head()

In [ ]:
print('ETH OHLCV DataFrame:')
eth_df.head()

### 1.3 Price Charts with Plotly

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('BTC/USD (Synthetic)', 'ETH/USD (Synthetic)'),
                    vertical_spacing=0.08)

fig.add_trace(go.Candlestick(
    x=btc_df.index, open=btc_df['Open'], high=btc_df['High'],
    low=btc_df['Low'], close=btc_df['Close'], name='BTC'
), row=1, col=1)

fig.add_trace(go.Candlestick(
    x=eth_df.index, open=eth_df['Open'], high=eth_df['High'],
    low=eth_df['Low'], close=eth_df['Close'], name='ETH'
), row=2, col=1)

fig.update_layout(height=800, title_text='Synthetic Cryptocurrency Price Data (2 Years)',
                  showlegend=False, xaxis_rangeslider_visible=False,
                  xaxis2_rangeslider_visible=False)
fig.show()

---
# Part 2: Technical Indicators

Technical analysis uses historical price and volume data to identify patterns and trading signals. We implement each indicator **from scratch** to understand the underlying mathematics.

We will use BTC data for all indicator demonstrations.

### 2.1 Simple Moving Average (SMA)

The SMA over $n$ periods is simply:

$$\text{SMA}_t = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}$$

Traders watch for **crossovers**: when a short-term SMA crosses above a long-term SMA, it is considered a bullish signal (and vice versa).

In [ ]:
def sma(series, window):
    """Simple Moving Average from scratch."""
    result = np.full(len(series), np.nan)
    for i in range(window - 1, len(series)):
        result[i] = np.mean(series[i - window + 1 : i + 1])
    return pd.Series(result, index=series.index, name=f'SMA_{window}')

btc_df['SMA_20'] = sma(btc_df['Close'], 20)
btc_df['SMA_50'] = sma(btc_df['Close'], 50)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(btc_df.index, btc_df['Close'], label='BTC Close', alpha=0.7)
ax.plot(btc_df.index, btc_df['SMA_20'], label='SMA 20', linewidth=1.5)
ax.plot(btc_df.index, btc_df['SMA_50'], label='SMA 50', linewidth=1.5)
ax.set_title('BTC Price with Simple Moving Averages')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### 2.2 Exponential Moving Average (EMA)

The EMA places greater weight on recent prices. The smoothing factor is $\alpha = \frac{2}{n+1}$:

$$\text{EMA}_t = \alpha \cdot P_t + (1 - \alpha) \cdot \text{EMA}_{t-1}$$

In [ ]:
def ema(series, span):
    """Exponential Moving Average from scratch."""
    alpha = 2.0 / (span + 1)
    result = np.full(len(series), np.nan)
    # Initialize with the first value
    result[0] = series.iloc[0]
    for i in range(1, len(series)):
        result[i] = alpha * series.iloc[i] + (1 - alpha) * result[i - 1]
    return pd.Series(result, index=series.index, name=f'EMA_{span}')

btc_df['EMA_12'] = ema(btc_df['Close'], 12)
btc_df['EMA_26'] = ema(btc_df['Close'], 26)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(btc_df.index, btc_df['Close'], label='BTC Close', alpha=0.6)
ax.plot(btc_df.index, btc_df['EMA_12'], label='EMA 12', linewidth=1.5)
ax.plot(btc_df.index, btc_df['EMA_26'], label='EMA 26', linewidth=1.5)
ax.set_title('BTC Price with Exponential Moving Averages')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### 2.3 Relative Strength Index (RSI)

RSI measures the speed and magnitude of price changes on a 0--100 scale:

$$\text{RSI} = 100 - \frac{100}{1 + RS}, \quad RS = \frac{\text{Avg Gain}}{\text{Avg Loss}}$$

Readings above 70 suggest **overbought** conditions; below 30 suggest **oversold**.

In [ ]:
def rsi(series, period=14):
    """Relative Strength Index from scratch using Wilder's smoothing."""
    delta = series.diff()
    gains = delta.clip(lower=0)
    losses = (-delta).clip(lower=0)
    
    result = np.full(len(series), np.nan)
    
    # Initial averages: simple mean over first `period` changes
    avg_gain = gains.iloc[1:period + 1].mean()
    avg_loss = losses.iloc[1:period + 1].mean()
    
    if avg_loss == 0:
        result[period] = 100.0
    else:
        result[period] = 100.0 - 100.0 / (1.0 + avg_gain / avg_loss)
    
    # Wilder's smoothing for subsequent values
    for i in range(period + 1, len(series)):
        avg_gain = (avg_gain * (period - 1) + gains.iloc[i]) / period
        avg_loss = (avg_loss * (period - 1) + losses.iloc[i]) / period
        if avg_loss == 0:
            result[i] = 100.0
        else:
            result[i] = 100.0 - 100.0 / (1.0 + avg_gain / avg_loss)
    
    return pd.Series(result, index=series.index, name='RSI')

btc_df['RSI'] = rsi(btc_df['Close'], 14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

ax1.plot(btc_df.index, btc_df['Close'], label='BTC Close')
ax1.set_title('BTC Price and RSI (14-day)')
ax1.set_ylabel('Price (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend()

ax2.plot(btc_df.index, btc_df['RSI'], color='purple', linewidth=1)
ax2.axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.fill_between(btc_df.index, 70, 100, alpha=0.1, color='red')
ax2.fill_between(btc_df.index, 0, 30, alpha=0.1, color='green')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 2.4 MACD (Moving Average Convergence Divergence)

MACD captures momentum through the relationship between two EMAs:

- **MACD Line** = EMA(12) - EMA(26)
- **Signal Line** = EMA(9) of the MACD Line
- **Histogram** = MACD Line - Signal Line

A bullish signal occurs when the MACD line crosses above the signal line.

In [ ]:
def macd(series, fast=12, slow=26, signal=9):
    """MACD from scratch."""
    ema_fast = ema(series, fast)
    ema_slow = ema(series, slow)
    macd_line = ema_fast - ema_slow
    macd_line.name = 'MACD'
    
    # Signal line: EMA of MACD line
    signal_line = ema(macd_line, signal)
    signal_line.name = 'Signal'
    
    histogram = macd_line - signal_line
    histogram.name = 'Histogram'
    
    return macd_line, signal_line, histogram

btc_df['MACD'], btc_df['MACD_Signal'], btc_df['MACD_Hist'] = macd(btc_df['Close'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

ax1.plot(btc_df.index, btc_df['Close'], label='BTC Close')
ax1.set_title('BTC Price and MACD (12, 26, 9)')
ax1.set_ylabel('Price (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend()

colors = ['green' if v >= 0 else 'red' for v in btc_df['MACD_Hist']]
ax2.bar(btc_df.index, btc_df['MACD_Hist'], color=colors, alpha=0.5, width=1, label='Histogram')
ax2.plot(btc_df.index, btc_df['MACD'], label='MACD', linewidth=1.2)
ax2.plot(btc_df.index, btc_df['MACD_Signal'], label='Signal', linewidth=1.2)
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.set_ylabel('MACD')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 2.5 Bollinger Bands

Bollinger Bands measure volatility-adjusted price levels:

- **Middle Band** = SMA(20)
- **Upper Band** = SMA(20) + 2 * std(20)
- **Lower Band** = SMA(20) - 2 * std(20)

Price touching the upper/lower band often signals potential reversal or continuation depending on market context.

In [ ]:
def bollinger_bands(series, window=20, num_std=2):
    """Bollinger Bands from scratch."""
    middle = np.full(len(series), np.nan)
    upper = np.full(len(series), np.nan)
    lower = np.full(len(series), np.nan)
    
    for i in range(window - 1, len(series)):
        window_data = series.iloc[i - window + 1 : i + 1]
        m = window_data.mean()
        s = window_data.std(ddof=0)
        middle[i] = m
        upper[i] = m + num_std * s
        lower[i] = m - num_std * s
    
    return (pd.Series(middle, index=series.index, name='BB_Mid'),
            pd.Series(upper, index=series.index, name='BB_Upper'),
            pd.Series(lower, index=series.index, name='BB_Lower'))

btc_df['BB_Mid'], btc_df['BB_Upper'], btc_df['BB_Lower'] = bollinger_bands(btc_df['Close'])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(btc_df.index, btc_df['Close'], label='BTC Close', alpha=0.8)
ax.plot(btc_df.index, btc_df['BB_Mid'], label='BB Middle (SMA 20)', linestyle='--', linewidth=1)
ax.plot(btc_df.index, btc_df['BB_Upper'], label='BB Upper', color='red', linewidth=0.8)
ax.plot(btc_df.index, btc_df['BB_Lower'], label='BB Lower', color='green', linewidth=0.8)
ax.fill_between(btc_df.index, btc_df['BB_Upper'], btc_df['BB_Lower'], alpha=0.1, color='blue')
ax.set_title('BTC Price with Bollinger Bands (20, 2)')
ax.set_ylabel('Price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

### 2.6 Combined Technical Dashboard

Let us combine all indicators into a single multi-panel dashboard for BTC.

In [ ]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    row_heights=[0.4, 0.2, 0.2, 0.2],
                    subplot_titles=('Price + Bollinger Bands + SMAs',
                                    'Volume', 'RSI (14)', 'MACD (12,26,9)'),
                    vertical_spacing=0.05)

# Row 1: Price with Bollinger Bands and SMAs
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['Close'], name='Close',
                         line=dict(color='black', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['BB_Upper'], name='BB Upper',
                         line=dict(color='rgba(255,0,0,0.3)', width=0.8)), row=1, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['BB_Lower'], name='BB Lower',
                         line=dict(color='rgba(0,128,0,0.3)', width=0.8),
                         fill='tonexty', fillcolor='rgba(100,100,255,0.05)'), row=1, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['SMA_20'], name='SMA 20',
                         line=dict(color='orange', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['SMA_50'], name='SMA 50',
                         line=dict(color='blue', width=1, dash='dot')), row=1, col=1)

# Row 2: Volume
fig.add_trace(go.Bar(x=btc_df.index, y=btc_df['Volume'], name='Volume',
                     marker_color='steelblue', opacity=0.5), row=2, col=1)

# Row 3: RSI
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['RSI'], name='RSI',
                         line=dict(color='purple', width=1)), row=3, col=1)
fig.add_hline(y=70, line_dash='dash', line_color='red', opacity=0.5, row=3, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', opacity=0.5, row=3, col=1)

# Row 4: MACD
hist_colors = ['green' if v >= 0 else 'red' for v in btc_df['MACD_Hist'].fillna(0)]
fig.add_trace(go.Bar(x=btc_df.index, y=btc_df['MACD_Hist'], name='Histogram',
                     marker_color=hist_colors, opacity=0.5), row=4, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['MACD'], name='MACD',
                         line=dict(color='blue', width=1)), row=4, col=1)
fig.add_trace(go.Scatter(x=btc_df.index, y=btc_df['MACD_Signal'], name='Signal',
                         line=dict(color='orange', width=1)), row=4, col=1)

fig.update_layout(height=1000, title_text='BTC Technical Analysis Dashboard',
                  showlegend=True, legend=dict(x=1.01, y=1))
fig.show()

---
# Part 3: On-Chain Metrics

On-chain metrics analyze data directly from the blockchain to assess network health and valuation. Unlike technical indicators that rely solely on price/volume, on-chain metrics incorporate transaction activity, active addresses, and other network-level data.

We generate synthetic on-chain data that mirrors the statistical properties of real blockchain data.

### 3.1 NVT Ratio (Network Value to Transactions)

The NVT Ratio is often called the "PE ratio of crypto":

$$\text{NVT} = \frac{\text{Market Cap}}{\text{Daily Transaction Volume (USD)}}$$

- **NVT < 20**: Network may be undervalued relative to its usage
- **NVT 20--65**: Normal range
- **NVT > 65**: Network may be overvalued (speculative bubble territory)

In [ ]:
# Synthetic circulating supply (slowly increasing, as in real BTC mining)
rng3 = np.random.default_rng(100)
btc_supply = 19_500_000 + np.cumsum(np.ones(len(btc_df)) * (900 / 30))  # ~900 BTC/month mined

# Market cap
btc_df['MarketCap'] = btc_df['Close'] * btc_supply

# Synthetic daily transaction volume (USD)
# Correlated with price but with additional noise and mean-reversion
base_tx_vol = btc_df['Close'] * 500  # rough baseline
noise = rng3.lognormal(0, 0.5, size=len(btc_df))
btc_df['TxVolume'] = base_tx_vol * noise

# NVT Ratio
btc_df['NVT'] = btc_df['MarketCap'] / btc_df['TxVolume']

# Smoothed NVT (14-day SMA for cleaner signal)
btc_df['NVT_Smoothed'] = btc_df['NVT'].rolling(14).mean()

print(f"NVT Ratio statistics:")
print(btc_df['NVT'].describe())

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 1]}, sharex=True)

ax1.plot(btc_df.index, btc_df['Close'], color='black', linewidth=1)
ax1.set_title('BTC Price vs NVT Ratio')
ax1.set_ylabel('Price (USD)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

ax2.plot(btc_df.index, btc_df['NVT_Smoothed'], color='darkorange', linewidth=1.2, label='NVT (14d SMA)')
ax2.axhline(65, color='red', linestyle='--', alpha=0.7, label='Overvalued zone (>65)')
ax2.axhline(20, color='green', linestyle='--', alpha=0.7, label='Undervalued zone (<20)')
ax2.fill_between(btc_df.index, 65, btc_df['NVT_Smoothed'].max() * 1.1, alpha=0.08, color='red')
ax2.fill_between(btc_df.index, 0, 20, alpha=0.08, color='green')
ax2.set_ylabel('NVT Ratio')
ax2.set_ylim(0, btc_df['NVT_Smoothed'].quantile(0.99) * 1.2)
ax2.legend()

plt.tight_layout()
plt.show()

### 3.2 Active Addresses (Synthetic Proxy)

Active addresses measure the number of unique addresses participating in transactions on a given day. Higher activity generally signals greater network utility.

In [ ]:
# Synthetic active addresses: correlated with price and with mean-reversion
rng4 = np.random.default_rng(200)
log_price_norm = (np.log(btc_df['Close']) - np.log(btc_df['Close']).mean()) / np.log(btc_df['Close']).std()

base_addresses = 800_000
price_effect = 150_000 * log_price_norm.values
noise_addresses = rng4.normal(0, 50_000, size=len(btc_df))

btc_df['ActiveAddresses'] = np.maximum(base_addresses + price_effect + noise_addresses, 300_000).astype(int)

fig, ax1 = plt.subplots(figsize=(14, 6))
color1 = 'tab:blue'
ax1.plot(btc_df.index, btc_df['Close'], color=color1, alpha=0.8, label='BTC Price')
ax1.set_ylabel('Price (USD)', color=color1)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tab:orange'
ax2.plot(btc_df.index, btc_df['ActiveAddresses'].rolling(7).mean(), color=color2, alpha=0.8, label='Active Addresses (7d avg)')
ax2.set_ylabel('Active Addresses', color=color2)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.tick_params(axis='y', labelcolor=color2)

ax1.set_title('BTC Price vs Active Addresses')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

### 3.3 On-Chain Metrics Comparison

Let us visualize how on-chain signals relate to price movements.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price
axes[0].plot(btc_df.index, btc_df['Close'], color='black', linewidth=1)
axes[0].set_ylabel('Price (USD)')
axes[0].set_title('BTC: Price, NVT, and Active Addresses')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# NVT
axes[1].plot(btc_df.index, btc_df['NVT_Smoothed'], color='darkorange', linewidth=1)
axes[1].axhline(65, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(20, color='green', linestyle='--', alpha=0.5)
axes[1].set_ylabel('NVT (14d)')
axes[1].set_ylim(0, btc_df['NVT_Smoothed'].quantile(0.99) * 1.2)

# Active Addresses
axes[2].plot(btc_df.index, btc_df['ActiveAddresses'].rolling(7).mean(), color='steelblue', linewidth=1)
axes[2].set_ylabel('Active Addr (7d)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

---
# Part 4: Correlation Analysis

Understanding how crypto assets co-move is essential for portfolio construction, hedging, and risk management. We now generate price data for five assets and analyze their correlation structure.

### 4.1 Generating Multi-Asset Price Data

In [ ]:
# Asset parameters: (name, S0, mu, sigma)
assets = {
    'BTC':  {'S0': 30000, 'mu': 0.15, 'sigma': 0.60},
    'ETH':  {'S0': 2000,  'mu': 0.20, 'sigma': 0.80},
    'SOL':  {'S0': 25,    'mu': 0.30, 'sigma': 1.00},
    'ADA':  {'S0': 0.40,  'mu': 0.10, 'sigma': 0.90},
    'SPY':  {'S0': 450,   'mu': 0.10, 'sigma': 0.18},  # Traditional asset proxy (S&P 500 ETF)
}

# Correlation matrix (designed to reflect realistic cross-asset relationships)
asset_names = list(assets.keys())
corr_matrix = np.array([
    # BTC   ETH   SOL   ADA   SPY
    [1.00, 0.70, 0.60, 0.55, 0.30],  # BTC
    [0.70, 1.00, 0.65, 0.60, 0.25],  # ETH
    [0.60, 0.65, 1.00, 0.50, 0.20],  # SOL
    [0.55, 0.60, 0.50, 1.00, 0.15],  # ADA
    [0.30, 0.25, 0.20, 0.15, 1.00],  # SPY
])

# Cholesky decomposition for correlated normals
L = np.linalg.cholesky(corr_matrix)
rng5 = np.random.default_rng(55)
Z_ind = rng5.standard_normal((n_days, len(asset_names)))
Z_corr = Z_ind @ L.T

# Generate prices
prices = {}
for i, name in enumerate(asset_names):
    p = assets[name]
    drift = (p['mu'] - 0.5 * p['sigma']**2) * dt
    log_ret = drift + p['sigma'] * np.sqrt(dt) * Z_corr[:, i]
    price_series = p['S0'] * np.exp(np.concatenate([[0], np.cumsum(log_ret)]))
    prices[name] = price_series

dates = pd.date_range(start='2024-01-01', periods=n_days + 1, freq='D')
price_df = pd.DataFrame(prices, index=dates)
price_df.index.name = 'Date'

print('Multi-asset price data generated.')
price_df.head()

### 4.2 Daily Returns

In [ ]:
# Log returns
returns_df = np.log(price_df / price_df.shift(1)).dropna()

print('Daily Log Returns Summary:')
print(returns_df.describe())

fig, axes = plt.subplots(1, len(asset_names), figsize=(18, 4), sharey=True)
for i, name in enumerate(asset_names):
    axes[i].hist(returns_df[name], bins=50, alpha=0.7, color=sns.color_palette()[i], edgecolor='white')
    axes[i].set_title(f'{name} Daily Returns')
    axes[i].axvline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Frequency')
plt.suptitle('Distribution of Daily Log Returns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 4.3 Correlation Matrix and Heatmap

In [ ]:
realized_corr = returns_df.corr()

print('Realized Return Correlation Matrix:')
print(realized_corr.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(realized_corr, dtype=bool), k=1)
sns.heatmap(realized_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, mask=mask, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Realized Correlation of Daily Returns')
plt.tight_layout()
plt.show()

### 4.4 Rolling Correlation: BTC vs ETH

Correlations are not static. A rolling window reveals how the BTC-ETH relationship evolves over time.

In [ ]:
rolling_corr_30 = returns_df['BTC'].rolling(30).corr(returns_df['ETH'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={'height_ratios': [1, 1]})

# Normalized prices for comparison
ax1.plot(price_df.index, price_df['BTC'] / price_df['BTC'].iloc[0], label='BTC (normalized)')
ax1.plot(price_df.index, price_df['ETH'] / price_df['ETH'].iloc[0], label='ETH (normalized)')
ax1.set_title('BTC vs ETH: Normalized Price and Rolling 30-day Correlation')
ax1.set_ylabel('Normalized Price')
ax1.legend()

ax2.plot(rolling_corr_30.index, rolling_corr_30, color='purple', linewidth=1)
ax2.axhline(0.7, color='gray', linestyle='--', alpha=0.5, label='Target r=0.7')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.fill_between(rolling_corr_30.index, rolling_corr_30, 0, alpha=0.2, color='purple')
ax2.set_ylabel('30-day Rolling Correlation')
ax2.set_ylim(-0.5, 1.0)
ax2.legend()

plt.tight_layout()
plt.show()

### 4.5 Beta Calculation

Beta measures the sensitivity of an asset's returns to a benchmark (here, BTC as the market proxy):

$$\beta_i = \frac{\text{Cov}(R_i, R_{\text{BTC}})}{\text{Var}(R_{\text{BTC}})}$$

A beta greater than 1 means the asset amplifies BTC's moves; less than 1 means it dampens them.

In [ ]:
btc_returns = returns_df['BTC']
btc_var = btc_returns.var()

betas = {}
for name in asset_names:
    cov = returns_df[name].cov(btc_returns)
    betas[name] = cov / btc_var

beta_df = pd.DataFrame.from_dict(betas, orient='index', columns=['Beta vs BTC'])
beta_df['Ann. Volatility'] = returns_df.std() * np.sqrt(365)
beta_df['Ann. Return'] = returns_df.mean() * 365

print('Beta and Risk-Return Profile (BTC as benchmark):')
print(beta_df.round(4))

fig, ax = plt.subplots(figsize=(8, 6))
colors = sns.color_palette('deep', len(asset_names))
for i, name in enumerate(asset_names):
    ax.scatter(beta_df.loc[name, 'Ann. Volatility'], beta_df.loc[name, 'Ann. Return'],
              s=150, color=colors[i], zorder=5, edgecolors='black')
    ax.annotate(name, (beta_df.loc[name, 'Ann. Volatility'], beta_df.loc[name, 'Ann. Return']),
               textcoords='offset points', xytext=(10, 5), fontsize=12, fontweight='bold')

ax.set_xlabel('Annualized Volatility')
ax.set_ylabel('Annualized Return')
ax.set_title('Risk-Return Profile of Crypto Assets')
ax.axhline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

---
# Part 5: Portfolio Analysis

Modern Portfolio Theory (Markowitz, 1952) shows that diversification can improve risk-adjusted returns. We use Monte Carlo simulation to map the **efficient frontier** -- the set of portfolios offering the highest return for each level of risk.

We work with the four crypto assets (BTC, ETH, SOL, ADA) and the SPY proxy.

### 5.1 Portfolio Return and Risk

For a portfolio with weights $\mathbf{w}$:

$$R_p = \mathbf{w}^T \boldsymbol{\mu}, \quad \sigma_p = \sqrt{\mathbf{w}^T \Sigma \mathbf{w}}$$

The **Sharpe Ratio** measures risk-adjusted return:

$$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$$

where $R_f = 4\%$ (annualized risk-free rate).

In [ ]:
# Annualized statistics
mean_returns = returns_df.mean() * 365
cov_matrix = returns_df.cov() * 365
risk_free_rate = 0.04
n_assets = len(asset_names)

print('Annualized Expected Returns:')
print(mean_returns.round(4))
print('\nAnnualized Covariance Matrix:')
print(cov_matrix.round(4))

### 5.2 Monte Carlo Efficient Frontier

In [ ]:
n_portfolios = 5000
rng6 = np.random.default_rng(99)

results = np.zeros((n_portfolios, 3 + n_assets))  # return, vol, sharpe, weights...

for i in tqdm(range(n_portfolios), desc='Simulating portfolios'):
    # Random weights (Dirichlet distribution for proper simplex sampling)
    w = rng6.dirichlet(np.ones(n_assets))
    
    # Portfolio return
    port_return = np.dot(w, mean_returns)
    
    # Portfolio volatility
    port_vol = np.sqrt(np.dot(w.T, np.dot(cov_matrix.values, w)))
    
    # Sharpe ratio
    sharpe = (port_return - risk_free_rate) / port_vol
    
    results[i, 0] = port_return
    results[i, 1] = port_vol
    results[i, 2] = sharpe
    results[i, 3:] = w

results_df = pd.DataFrame(results,
    columns=['Return', 'Volatility', 'Sharpe'] + [f'w_{n}' for n in asset_names])

print(f'Simulated {n_portfolios} portfolios.')
print(f'Sharpe ratio range: {results_df["Sharpe"].min():.3f} to {results_df["Sharpe"].max():.3f}')

### 5.3 Plotting the Efficient Frontier

In [ ]:
# Identify optimal portfolios
max_sharpe_idx = results_df['Sharpe'].idxmax()
min_vol_idx = results_df['Volatility'].idxmin()

max_sharpe_port = results_df.loc[max_sharpe_idx]
min_vol_port = results_df.loc[min_vol_idx]

fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(results_df['Volatility'], results_df['Return'],
                     c=results_df['Sharpe'], cmap='viridis',
                     s=8, alpha=0.6, edgecolors='none')

# Max Sharpe portfolio
ax.scatter(max_sharpe_port['Volatility'], max_sharpe_port['Return'],
           marker='*', color='red', s=400, zorder=5, edgecolors='black',
           label=f'Max Sharpe ({max_sharpe_port["Sharpe"]:.3f})')

# Min Volatility portfolio
ax.scatter(min_vol_port['Volatility'], min_vol_port['Return'],
           marker='D', color='blue', s=150, zorder=5, edgecolors='black',
           label=f'Min Volatility ({min_vol_port["Volatility"]:.3f})')

# Individual assets
for i, name in enumerate(asset_names):
    ax.scatter(np.sqrt(cov_matrix.loc[name, name]), mean_returns[name],
              marker='o', s=100, zorder=5, edgecolors='black',
              color=sns.color_palette('Set2')[i])
    ax.annotate(name, (np.sqrt(cov_matrix.loc[name, name]), mean_returns[name]),
               textcoords='offset points', xytext=(8, 5), fontsize=11, fontweight='bold')

plt.colorbar(scatter, ax=ax, label='Sharpe Ratio')
ax.set_xlabel('Annualized Volatility', fontsize=12)
ax.set_ylabel('Annualized Return', fontsize=12)
ax.set_title('Efficient Frontier (Monte Carlo, 5000 Portfolios)', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 5.4 Optimal Portfolio Composition

In [ ]:
print('=== Maximum Sharpe Ratio Portfolio ===')
print(f'Expected Return : {max_sharpe_port["Return"]:.2%}')
print(f'Volatility      : {max_sharpe_port["Volatility"]:.2%}')
print(f'Sharpe Ratio    : {max_sharpe_port["Sharpe"]:.3f}')
print('\nWeights:')
for name in asset_names:
    print(f'  {name:4s}: {max_sharpe_port[f"w_{name}"]:.1%}')

print('\n=== Minimum Volatility Portfolio ===')
print(f'Expected Return : {min_vol_port["Return"]:.2%}')
print(f'Volatility      : {min_vol_port["Volatility"]:.2%}')
print(f'Sharpe Ratio    : {min_vol_port["Sharpe"]:.3f}')
print('\nWeights:')
for name in asset_names:
    print(f'  {name:4s}: {min_vol_port[f"w_{name}"]:.1%}')

# Pie chart of Max Sharpe portfolio
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
weights_sharpe = [max_sharpe_port[f'w_{n}'] for n in asset_names]
weights_minvol = [min_vol_port[f'w_{n}'] for n in asset_names]

ax1.pie(weights_sharpe, labels=asset_names, autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('Set2', n_assets))
ax1.set_title('Max Sharpe Portfolio')

ax2.pie(weights_minvol, labels=asset_names, autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('Set2', n_assets))
ax2.set_title('Min Volatility Portfolio')

plt.suptitle('Optimal Portfolio Allocations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
# Exercises

The following exercises reinforce the concepts from this notebook. For theoretical context on crypto-asset valuation and market efficiency, refer to `sections/04-blockchain-economics.md`.

## Exercise 1: SMA Crossover Backtest

Implement a simple trading strategy based on SMA crossovers:
- **Buy signal**: SMA(20) crosses above SMA(50)
- **Sell signal**: SMA(20) crosses below SMA(50)
- Start with $10,000
- Calculate total return, max drawdown, and number of trades
- Compare to buy-and-hold

*Hint: Use `np.sign()` to detect crossover points where the sign of (SMA_20 - SMA_50) changes.*

In [ ]:
# Exercise 1: YOUR CODE HERE
# -------------------------------------------------
# Step 1: Compute SMA_20 and SMA_50 (already in btc_df)
# Step 2: Generate buy/sell signals from crossovers
# Step 3: Simulate portfolio value over time
# Step 4: Calculate performance metrics
# Step 5: Plot strategy vs buy-and-hold
# -------------------------------------------------


## Exercise 2: Correlation Dashboard for 5+ Assets

Create an interactive Plotly dashboard that displays:
1. A heatmap of the rolling 60-day correlation matrix for all 5 assets
2. Rolling correlation time series for all pairs vs BTC
3. A scatter matrix of returns

*Hint: Use `plotly.figure_factory.create_annotated_heatmap()` and `plotly.express.scatter_matrix()`.*

In [ ]:
# Exercise 2: YOUR CODE HERE
# -------------------------------------------------
# Step 1: Compute rolling 60-day correlations for all pairs
# Step 2: Build a multi-panel Plotly figure
# Step 3: Add scatter matrix of returns
# -------------------------------------------------


## Exercise 3: Risk-Adjusted Returns for Different Allocations

Compare these three allocation strategies over the 2-year period:
1. **Equal-weight**: 20% in each of the 5 assets
2. **BTC-heavy**: 60% BTC, 10% each for the other four
3. **Diversified**: 30% SPY, 25% BTC, 20% ETH, 15% SOL, 10% ADA

For each, calculate:
- Cumulative return
- Annualized volatility
- Sharpe ratio (Rf = 4%)
- Maximum drawdown
- Sortino ratio

Plot the equity curves on the same chart.

In [ ]:
# Exercise 3: YOUR CODE HERE
# -------------------------------------------------
# Step 1: Define the three weight vectors
# Step 2: Compute daily portfolio returns for each
# Step 3: Build cumulative equity curves
# Step 4: Calculate all performance metrics
# Step 5: Plot and create a summary table
# -------------------------------------------------


## Exercise 4: NVT-Based Signal Analysis

Build a simple NVT-based trading signal:
- When smoothed NVT drops below 25, mark as **Accumulate** (green zone)
- When smoothed NVT rises above 60, mark as **Distribute** (red zone)
- Otherwise, **Hold** (neutral)

Tasks:
1. Apply these rules to generate daily signals
2. Calculate the average forward 30-day return for each signal type
3. Create a visualization showing price with colored background regions for each signal
4. Discuss limitations of NVT as a standalone signal

*This exercise connects to the valuation frameworks in `sections/04-blockchain-economics.md`.*

In [ ]:
# Exercise 4: YOUR CODE HERE
# -------------------------------------------------
# Step 1: Create signal column based on NVT_Smoothed thresholds
# Step 2: Calculate forward 30-day returns for each signal
# Step 3: Visualize price with colored signal zones
# Step 4: Print summary statistics per signal
# -------------------------------------------------


---
# Summary

In this notebook we covered the core quantitative tools for cryptocurrency market analysis:

**Part 1 -- Synthetic Data Generation**
- Used Geometric Brownian Motion with Cholesky-correlated innovations to generate realistic price data
- Built complete OHLCV DataFrames with plausible intraday ranges and volume profiles

**Part 2 -- Technical Indicators**
- Implemented SMA, EMA, RSI, MACD, and Bollinger Bands entirely from scratch
- Each indicator captures a different dimension of price dynamics (trend, momentum, volatility)
- Combined indicators into a unified dashboard for holistic analysis

**Part 3 -- On-Chain Metrics**
- Calculated the NVT Ratio as a fundamental valuation tool
- Generated synthetic active address data and compared on-chain metrics to price
- Discussed interpretation zones for NVT signals

**Part 4 -- Correlation Analysis**
- Analyzed cross-asset correlations using return-based measures
- Demonstrated that correlations are time-varying with rolling window analysis
- Calculated beta values to measure systematic risk exposure

**Part 5 -- Portfolio Optimization**
- Mapped the efficient frontier via Monte Carlo simulation of 5,000 random portfolios
- Identified maximum Sharpe ratio and minimum volatility portfolios
- Illustrated how diversification across crypto and traditional assets improves risk-adjusted returns

## Key Takeaways
1. Technical analysis provides **short-term signals** but should be combined with fundamental/on-chain analysis
2. Crypto asset correlations are **higher than traditional markets** but vary significantly over time
3. Portfolio diversification delivers meaningful benefits even within the crypto asset class
4. All models are approximations -- GBM does not capture fat tails, jumps, or regime changes present in real crypto markets

## Further Reading
- See `sections/04-blockchain-economics.md` for theoretical foundations on token valuation and market efficiency
- Markowitz, H. (1952). "Portfolio Selection." *The Journal of Finance*.
- Woo, W. (2017). "Bitcoin NVT Ratio." *Woobull Charts*.
- Burniske, C. & Tatar, J. (2018). *Cryptoassets: The Innovative Investor's Guide*. McGraw-Hill.